In [3]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

# Target log file (fixed from typo-like path in request)
LOG_PATH = Path("../../results/gpt64/lvb/agent_results.json")
BACKUP_PATH = LOG_PATH.with_suffix(LOG_PATH.suffix + ".bak")
TMP_PATH = LOG_PATH.with_suffix(LOG_PATH.suffix + ".compact_tmp")


def compact_message_for_log(message: dict[str, Any]) -> dict[str, Any]:
    role = message.get("role")
    content = message.get("content")

    if isinstance(content, str):
        return {"role": role, "content": content}

    if isinstance(content, list):
        text_parts: list[str] = []
        image_count = 0
        for item in content:
            if not isinstance(item, dict):
                continue
            if item.get("type") == "text":
                txt = item.get("text")
                if isinstance(txt, str):
                    text_parts.append(txt)
            elif item.get("type") == "image_url":
                image_count += 1
        return {
            "role": role,
            "content_text": "\n".join(text_parts).strip(),
            "image_count": image_count,
        }

    return {"role": role, "content": ""}


def compact_conversation_for_log(conversation: Any) -> Any:
    if not isinstance(conversation, list):
        return conversation
    compacted = []
    for msg in conversation:
        if isinstance(msg, dict):
            compacted.append(compact_message_for_log(msg))
        else:
            compacted.append(msg)
    return compacted


def compact_record(record: dict[str, Any]) -> dict[str, Any]:
    for key in ("pred", "all_preds"):
        value = record.get(key)
        if isinstance(value, list):
            record[key] = [compact_conversation_for_log(c) for c in value]

    pass_traces = record.get("pass_traces")
    if isinstance(pass_traces, list):
        for trace in pass_traces:
            if isinstance(trace, dict) and isinstance(trace.get("conversation"), list):
                trace["conversation"] = compact_conversation_for_log(trace["conversation"])
    return record


def stream_objects_from_json_array(path: Path, chunk_size: int = 8 * 1024 * 1024):
    decoder = json.JSONDecoder()
    with path.open("r", encoding="utf-8", errors="strict") as f:
        buf = ""
        pos = 0
        eof = False
        started = False

        while True:
            if not eof and (len(buf) - pos) < chunk_size // 2:
                more = f.read(chunk_size)
                if more == "":
                    eof = True
                else:
                    buf = buf[pos:] + more
                    pos = 0

            n = len(buf)
            while pos < n and buf[pos].isspace():
                pos += 1
            if pos >= n:
                if eof:
                    return
                continue

            if not started:
                if buf[pos] != "[":
                    raise ValueError("Expected '[' at start of JSON array")
                started = True
                pos += 1
                continue

            while pos < n and buf[pos].isspace():
                pos += 1
            if pos >= n:
                if eof:
                    return
                continue

            ch = buf[pos]
            if ch == "]":
                return
            if ch == ",":
                pos += 1
                continue

            try:
                obj, end = decoder.raw_decode(buf, pos)
            except json.JSONDecodeError:
                if eof:
                    # Truncated tail: keep valid prefix only
                    return
                break

            pos = end
            yield obj


assert LOG_PATH.exists(), f"Missing log file: {LOG_PATH}"

count = 0
with TMP_PATH.open("w", encoding="utf-8") as out:
    out.write("[\n")
    first = True
    for obj in stream_objects_from_json_array(LOG_PATH):
        if not isinstance(obj, dict):
            continue
        obj = compact_record(obj)
        if not first:
            out.write(",\n")
        json.dump(obj, out, ensure_ascii=False)
        first = False
        count += 1
    out.write("\n]\n")

orig_size = LOG_PATH.stat().st_size
new_size = TMP_PATH.stat().st_size

if BACKUP_PATH.exists():
    BACKUP_PATH.unlink()
os.replace(LOG_PATH, BACKUP_PATH)
os.replace(TMP_PATH, LOG_PATH)

print(f"Compacted {count} records")
print(f"Original size: {orig_size:,} bytes")
print(f"Compacted size: {new_size:,} bytes")
print(f"Backup: {BACKUP_PATH}")
print(f"Rewritten: {LOG_PATH}")


Compacted 2 records
Original size: 1,022,711,853 bytes
Compacted size: 12,789 bytes
Backup: ../../results/gpt64/lvb/agent_results.json.bak
Rewritten: ../../results/gpt64/lvb/agent_results.json
